# Notebook 4 — ConvNeXt-Tiny retrain (fair training budget)**Why.** The original ConvNeXt run stopped at **epoch 9 of 30**: `freeze_epochs=8`unfroze the backbone at epoch 9 and `patience=5` ended the run almost immediatelyafter, so the backbone was fine-tuned for roughly one epoch. Its final train/val gapof 0.0009 is an under-fitting signature, not an architectural limit(FACT_CHECK E5). Any claim about ConvNeXt's relative capability — and the wholeablation — rests on giving it a comparable budget.**This notebook is `src/models/convnext_tiny_5class.ipynb` with Kaggle paths andexactly three changes:**| Setting | Original | Here ||---|---|---|| `freeze_epochs` | 8 | **3** || `patience` | 5 | **10** || `num_epochs` | 30 | **40** |Everything else — seed 42, lr 2e-4, wd 3e-4, dropout 0.4, drop_path 0.2,label_smooth 0.15, batch 12, warmup 3, the boosts, the sampler, the compositecriterion, both transforms — is byte-identical to the original.**Outputs:** `convnext_tiny_5cls_skin_v2.pth` (same wrapped format + MCC),`history.csv`, `timing.json`, `results_convnext_v2.json`.Runtime on 1x T4: roughly 4-7 min/epoch, so budget ~3-4 h for the full 40 epochs.

In [ ]:
# ============================ EDIT THESE ============================DATA_ROOT = '/kaggle/input/datasets/jainmanav20/ipd-skin-dataset/IPD_Datasets_Final_Split'CKPT_ROOT = '/kaggle/input/datasets/jainmanav20/checkpoints'   # for the eval/ablation cellOUT_DIR   = '/kaggle/working'# ====================================================================!pip install -q timmimport os, random, warnings, json, timefrom pathlib import Pathfrom collections import Counterimport numpy as npimport pandas as pdimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport torchimport torch.nn as nnimport torch.nn.functional as Fimport torch.optim as optimfrom torch.utils.data import DataLoader, WeightedRandomSamplerfrom torchvision import transformsfrom torchvision.datasets import ImageFolderimport timmfrom sklearn.metrics import (    accuracy_score, classification_report, roc_auc_score, roc_curve, auc,    confusion_matrix, matthews_corrcoef, balanced_accuracy_score, f1_score,    precision_recall_fscore_support,)from sklearn.preprocessing import label_binarizewarnings.filterwarnings('ignore')# Seed 42 -- identical to the original run.SEED = 42random.seed(SEED); np.random.seed(SEED)torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU'print(f'Device : {DEVICE}')print(f'GPU    : {GPU_NAME}')if DEVICE.type == 'cuda':    props = torch.cuda.get_device_properties(0)    print(f'VRAM   : {props.total_memory/1024**3:.1f} GB')    torch.backends.cuda.matmul.allow_tf32 = True    torch.backends.cudnn.allow_tf32 = True    torch.backends.cudnn.benchmark = Trueprint(f'torch {torch.__version__} | timm {timm.__version__}')# ---------------------------------------------------------------------------# Accelerator guard. A CPU run of this notebook does not fit a Kaggle session:# the 2026-10-29 CPU attempt reached epoch 27 of 40 in 11.6 h and was killed at# the 12 h wall with no test evaluation and no verified checkpoint. Measured# step times were 3,825 ms/step unfrozen on CPU against 110 ms/step for# Swin-Tiny on one T4. Fail in seconds rather than after eleven hours.ALLOW_CPU = False   # set True only for a deliberate short smoke testif DEVICE.type != 'cuda' and not ALLOW_CPU:    raise RuntimeError(        'No CUDA device. This notebook needs a GPU.\n'        '  Fix: Settings -> Accelerator -> GPU T4 x1, '        'then Save Version -> Save & Run All.\n'        '  Expected on one T4: ~1-1.3 h for 40 epochs '        '(Swin-Tiny did 30 epochs in 0.59 h).\n'        '  Estimated on CPU  : ~17.6 h -> exceeds the 12 h session limit.\n'        '  Override with ALLOW_CPU = True only for a deliberate smoke test.'    )LOCAL_ROOT = Path(DATA_ROOT); SAVE_DIR = Path(OUT_DIR)SAVE_DIR.mkdir(parents=True, exist_ok=True)

## Config — only freeze_epochs / patience / num_epochs differ from the original

In [ ]:
CFG = dict(    model_name      = 'convnext_tiny',    img_size        = 224,    batch_size      = 12,    num_epochs      = 40,      # CHANGED from 30    lr              = 2e-4,    weight_decay    = 3e-4,    dropout         = 0.4,    drop_path_rate  = 0.2,    label_smooth    = 0.15,    warmup_epochs   = 3,    use_amp         = True,    num_workers     = 2,       # Kaggle gives more CPU than the original Colab box    freeze_epochs   = 3,       # CHANGED from 8    patience        = 10,      # CHANGED from 5    lr_decay_rate   = 0.1,    melanoma_boost  = 2.0,    bcc_boost       = 2.0,    tinea_boost     = 3.0,    eczema_boost    = 1.5,    ckpt_path       = str(SAVE_DIR/'best_convnext_tiny_5cls_v2.pth'),)ORIGINAL = dict(freeze_epochs=8, patience=5, num_epochs=30)print('Config (deltas vs original run):')for k, v in CFG.items():    tag = f'   <-- CHANGED from {ORIGINAL[k]}' if k in ORIGINAL and ORIGINAL[k] != v else ''    print(f'  {k:16s} = {v}{tag}')

## Transforms — verbatim from the original notebook

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]IMAGENET_STD  = [0.229, 0.224, 0.225]train_tf = transforms.Compose([    transforms.Resize((CFG['img_size'] + 32, CFG['img_size'] + 32)),    transforms.RandomCrop(CFG['img_size']),    transforms.RandomHorizontalFlip(),    transforms.RandomVerticalFlip(),    transforms.RandomRotation(30),    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10, scale=(0.85, 1.15)),    transforms.RandomGrayscale(p=0.05),    transforms.ToTensor(),    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2), ratio=(0.3, 3.3)),])val_tf = transforms.Compose([    transforms.Resize((CFG['img_size'], CFG['img_size'])),    transforms.ToTensor(),    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),])print('Transforms ready (identical to the original run).')

## Data — restricted to the 5 paper classes, in checkpoint orderThe corpus also holds Acne, Lichen_Planus, Psoriasis and Vitiligo; those folders areexcluded, and class order is pinned so indices match the existing checkpoints.

In [ ]:
CLASSES = ['Basal_Cell_Carcinoma', 'Eczema', 'Keratosis', 'Melanoma', 'Tinea']NUM_CLS = len(CLASSES)class FiveClassImageFolder(ImageFolder):    def find_classes(self, directory):        return list(CLASSES), {c: i for i, c in enumerate(CLASSES)}train_ds = FiveClassImageFolder(LOCAL_ROOT/'train', transform=train_tf)val_ds   = FiveClassImageFolder(LOCAL_ROOT/'val',   transform=val_tf)test_ds  = FiveClassImageFolder(LOCAL_ROOT/'test',  transform=val_tf)assert train_ds.classes == CLASSESprint(f'Classes ({NUM_CLS}): {CLASSES}')print(f'Train : {len(train_ds)} | Val : {len(val_ds)} | Test : {len(test_ds)}')class_counts = Counter(train_ds.targets)print('\nTraining class distribution:')for i, cls in enumerate(CLASSES):    print(f'  {cls:<25} {class_counts[i]:>5}')class_weights_sampler = 1.0 / torch.tensor(    [class_counts[i] for i in range(NUM_CLS)], dtype=torch.float)sample_w = [class_weights_sampler[l] for l in train_ds.targets]sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler,                          num_workers=CFG['num_workers'], pin_memory=True)val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,                          num_workers=CFG['num_workers'], pin_memory=True)test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,                          num_workers=CFG['num_workers'], pin_memory=True)print('DataLoaders ready.')

## Model — verbatim from the original notebook

In [ ]:
class ConvNeXtSkinClassifier(nn.Module):    def __init__(self, num_classes, dropout=0.4, drop_path_rate=0.2, pretrained=True):        super().__init__()        self.backbone = timm.create_model(            'convnext_tiny', pretrained=pretrained,            num_classes=0, drop_rate=dropout, drop_path_rate=drop_path_rate,        )        feat_dim = self.backbone.num_features  # 768        self.classifier = nn.Sequential(            nn.LayerNorm(feat_dim),            nn.Dropout(dropout),            nn.Linear(feat_dim, 256),            nn.GELU(),            nn.Dropout(dropout / 2),            nn.Linear(256, num_classes),        )        print(f'ConvNeXt-Tiny | features={feat_dim} | classes={num_classes} | '              f'dropout={dropout} | drop_path={drop_path_rate}')    def forward(self, x): return self.classifier(self.backbone(x))    def freeze_backbone(self):        for p in self.backbone.parameters(): p.requires_grad = False        n = sum(p.numel() for p in self.parameters() if p.requires_grad)        print(f'  Backbone frozen    | trainable: {n:,}')    def unfreeze_backbone(self):        for p in self.backbone.parameters(): p.requires_grad = True        n = sum(p.numel() for p in self.parameters() if p.requires_grad)        print(f'  Backbone unfrozen  | trainable: {n:,}')model = ConvNeXtSkinClassifier(    num_classes=NUM_CLS, dropout=CFG['dropout'],    drop_path_rate=CFG['drop_path_rate'], pretrained=True).to(DEVICE)model = model.to(memory_format=torch.channels_last)print(f'Total params    : {sum(p.numel() for p in model.parameters()):,}')

## Loss, optimiser, scheduler — verbatim

In [ ]:
counts_arr   = np.array([class_counts[i] for i in range(NUM_CLS)], dtype=np.float32)loss_weights = 1.0 / np.sqrt(counts_arr)loss_weights /= loss_weights.sum()melanoma_idx = CLASSES.index('Melanoma')bcc_idx      = CLASSES.index('Basal_Cell_Carcinoma')tinea_idx    = CLASSES.index('Tinea')eczema_idx   = CLASSES.index('Eczema')loss_weights[melanoma_idx] *= CFG['melanoma_boost']loss_weights[bcc_idx]      *= CFG['bcc_boost']loss_weights[tinea_idx]    *= CFG['tinea_boost']loss_weights[eczema_idx]   *= CFG['eczema_boost']loss_weights /= loss_weights.sum()weights_tensor = torch.tensor(loss_weights, dtype=torch.float32).to(DEVICE)criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=CFG['label_smooth'])print('Class weights for loss:')for i, cls in enumerate(CLASSES):    print(f'  {cls:<25} {loss_weights[i]:.4f}')LOSS_WEIGHTS = {CLASSES[i]: float(loss_weights[i]) for i in range(NUM_CLS)}backbone_params = [p for n_, p in model.named_parameters()                   if 'classifier' not in n_ and p.requires_grad]head_params     = [p for n_, p in model.named_parameters()                   if 'classifier' in n_ and p.requires_grad]optimizer = optim.AdamW([    {'params': backbone_params, 'lr': CFG['lr'] * CFG['lr_decay_rate']},    {'params': head_params,     'lr': CFG['lr']},], weight_decay=CFG['weight_decay'])print(f'\nLayer-wise LR: backbone={CFG["lr"]*CFG["lr_decay_rate"]:.2e} | head={CFG["lr"]:.2e}')def lr_lambda(epoch):    if epoch < CFG['warmup_epochs']:        return (epoch + 1) / CFG['warmup_epochs']    progress = (epoch - CFG['warmup_epochs']) / max(1, CFG['num_epochs'] - CFG['warmup_epochs'])    return 0.5 * (1 + np.cos(np.pi * progress))scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)scaler    = torch.cuda.amp.GradScaler(enabled=CFG['use_amp'])print('Optimiser / Scheduler ready.')

## Training loop — verbatim, plus per-epoch wall-clock logging

In [ ]:
def train_epoch(loader):    model.train()    total_loss, correct, total = 0.0, 0, 0    for imgs, lbls in loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        lbls = lbls.to(DEVICE, non_blocking=True)        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):            logits = model(imgs); loss = criterion(logits, lbls)        optimizer.zero_grad(set_to_none=True)        scaler.scale(loss).backward()        scaler.unscale_(optimizer)        nn.utils.clip_grad_norm_(model.parameters(), 1.0)        scaler.step(optimizer); scaler.update()        total_loss += loss.item()*imgs.size(0)        correct    += (logits.argmax(1)==lbls).sum().item()        total      += imgs.size(0)    return total_loss/total, correct/total@torch.no_grad()def val_epoch(loader):    model.eval()    total_loss, correct, total = 0.0, 0, 0    all_p, all_l = [], []    for imgs, lbls in loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        lbls = lbls.to(DEVICE, non_blocking=True)        with torch.cuda.amp.autocast(enabled=CFG['use_amp']):            logits = model(imgs); loss = criterion(logits, lbls)        total_loss += loss.item()*imgs.size(0)        correct    += (logits.argmax(1)==lbls).sum().item()        total      += imgs.size(0)        all_p.extend(logits.argmax(1).cpu().numpy()); all_l.extend(lbls.cpu().numpy())    all_p = np.array(all_p); all_l = np.array(all_l)    mel_tp = ((all_p==melanoma_idx)&(all_l==melanoma_idx)).sum()    mel_fn = ((all_p!=melanoma_idx)&(all_l==melanoma_idx)).sum()    bcc_tp = ((all_p==bcc_idx)&(all_l==bcc_idx)).sum()    bcc_fn = ((all_p!=bcc_idx)&(all_l==bcc_idx)).sum()    mel_sens = mel_tp/(mel_tp+mel_fn) if (mel_tp+mel_fn)>0 else 0.0    bcc_sens = bcc_tp/(bcc_tp+bcc_fn) if (bcc_tp+bcc_fn)>0 else 0.0    return total_loss/total, correct/total, mel_sens, bcc_senshistory = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],           'gap': [], 'melanoma_sensitivity': [], 'bcc_sensitivity': [], 'composite': [],           'epoch_seconds': [], 'lr_head': []}best_composite, patience_cnt = 0.0, 0# Recorded at the SAVED epoch, not as a max() over all epochs. The original# notebook stored max(...) for these, so best_val_acc could come from a# different epoch than the weights that actually shipped.saved_epoch, saved_val_acc, saved_mel_sens, saved_bcc_sens = 0, 0.0, 0.0, 0.0t_start = time.time()print(f'\n{"Ep":>4} {"TrLoss":>8} {"VlLoss":>8} {"TrAcc":>7} {"VlAcc":>7} '      f'{"MelSens":>8} {"BCCSens":>8} {"Comp":>7} {"LR":>9} {"sec":>6}')print('-'*94)for epoch in range(1, CFG['num_epochs'] + 1):    t_ep = time.time()    if epoch == 1:        model.freeze_backbone()    elif epoch == CFG['freeze_epochs'] + 1:        model.unfreeze_backbone()        backbone_params = [p for n_, p in model.named_parameters() if 'classifier' not in n_]        head_params     = [p for n_, p in model.named_parameters() if 'classifier' in n_]        optimizer = optim.AdamW([            {'params': backbone_params, 'lr': CFG['lr'] * CFG['lr_decay_rate']},            {'params': head_params,     'lr': CFG['lr']},        ], weight_decay=CFG['weight_decay'])        scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)        for _ in range(epoch-1):            scheduler.step()        print('  -> Full fine-tuning with layer-wise LR.')    tr_loss, tr_acc               = train_epoch(train_loader)    vl_loss, vl_acc, mel_s, bcc_s = val_epoch(val_loader)    scheduler.step()    composite = 0.4*mel_s + 0.3*bcc_s + 0.3*vl_acc    gap = tr_acc - vl_acc    dt = time.time() - t_ep    current_lr = optimizer.param_groups[-1]['lr']    history['train_loss'].append(tr_loss);  history['val_loss'].append(vl_loss)    history['train_acc'].append(tr_acc);    history['val_acc'].append(vl_acc)    history['gap'].append(gap)    history['melanoma_sensitivity'].append(float(mel_s))    history['bcc_sensitivity'].append(float(bcc_s))    history['composite'].append(float(composite))    history['epoch_seconds'].append(dt);    history['lr_head'].append(current_lr)    tag = ''    if composite > best_composite:        best_composite = composite        torch.save(model.state_dict(), CFG['ckpt_path'])        patience_cnt = 0; tag = ' *'        saved_epoch, saved_val_acc = epoch, vl_acc        saved_mel_sens, saved_bcc_sens = float(mel_s), float(bcc_s)    else:        patience_cnt += 1    gap_flag = ' !!' if gap > 0.10 else (' !' if gap > 0.05 else '')    print(f'{epoch:>4} {tr_loss:>8.4f} {vl_loss:>8.4f} {tr_acc*100:>6.2f}% {vl_acc*100:>6.2f}% '          f'{mel_s*100:>7.2f}% {bcc_s*100:>7.2f}% {composite:>7.4f} '          f'{current_lr:>9.2e} {dt:>6.0f}{gap_flag}{tag}')    pd.DataFrame(history).assign(epoch=range(1, len(history['train_loss'])+1)) \      .to_csv(SAVE_DIR/'history.csv', index=False)    if patience_cnt >= CFG['patience']:        print(f'\nEarly stopping at epoch {epoch}.')        breaktotal_seconds = time.time() - t_startepochs_run = len(history['train_loss'])# src/models/CLAUDE.md documents best_val_acc as "val_acc of the# best-composite epoch". The originals used max(history[...]) instead,# which is not a property of the shipped weights. Use the saved epoch.best_val_acc  = saved_val_accbest_mel_sens = saved_mel_sensmax_val_acc_any_epoch = max(history['val_acc'])   # comparison onlyprint(f'\nEpochs run       : {epochs_run} / {CFG["num_epochs"]}')print(f'Best composite   : {best_composite:.4f}  (epoch {saved_epoch})')print(f'Saved-epoch val  : {best_val_acc*100:.2f}%')print(f'Saved-epoch mel  : {best_mel_sens*100:.2f}%  (target >= 95%)')print(f'Max val any epoch: {max_val_acc_any_epoch*100:.2f}%')print(f'Avg gap (last 5) : {np.mean(history["gap"][-5:])*100:.2f}%')print(f'Total time       : {total_seconds/3600:.2f} h '      f'({np.mean(history["epoch_seconds"]):.0f}s/epoch)')print('\nOriginal run for comparison: 9 epochs, best_val_acc 75.57%, '      'test_acc 71.35%, final gap 0.09%')with open(SAVE_DIR/'timing.json','w',encoding='utf-8') as f:    json.dump(dict(gpu=GPU_NAME, torch=torch.__version__, timm=timm.__version__,                   epochs_run=epochs_run, epochs_configured=CFG['num_epochs'],                   total_seconds=total_seconds, total_hours=total_seconds/3600,                   mean_epoch_seconds=float(np.mean(history['epoch_seconds'])),                   per_epoch_seconds=history['epoch_seconds'],                   batch_size=CFG['batch_size'],                   vram_gb=(float(torch.cuda.get_device_properties(0)                                  .total_memory/1024**3)                            if DEVICE.type == 'cuda' else None),                   # amp_requested is the config flag; amp_effective is                   # whether autocast actually did anything.                   # torch.cuda.amp.autocast is a no-op on CPU, so the two                   # differ on a CPU run and the single 'amp' field the                   # originals wrote was ambiguous.                   amp_requested=bool(CFG['use_amp']),                   amp_effective=bool(CFG['use_amp']                                      and DEVICE.type == 'cuda'),                   precision=('mixed (fp16 autocast + GradScaler)'                              if (CFG['use_amp'] and DEVICE.type == 'cuda')                              else 'fp32'),                   saved_epoch=int(saved_epoch),                   note=(f'Measured on {GPU_NAME} ({DEVICE.type}). Timing '                         f'for THIS ConvNeXt retrain only -- the paper may '                         f'not generalise it to the other two backbones. '                         f'Mixed precision '                         f'{"active" if (CFG["use_amp"] and DEVICE.type == "cuda") else "NOT active"}.')),              f, indent=2)print('wrote history.csv, timing.json')

## Evaluate the retrained model and save the checkpoint (same wrapped format + MCC)

In [ ]:
model.load_state_dict(torch.load(CFG['ckpt_path'], map_location=DEVICE))model.eval()all_preds, all_labels, all_probs = [], [], []with torch.no_grad():    for imgs, lbls in test_loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        logits = model(imgs)        all_probs.extend(torch.softmax(logits, dim=1).float().cpu().numpy())        all_preds.extend(logits.argmax(1).cpu().numpy())        all_labels.extend(lbls.numpy())all_preds = np.array(all_preds); all_labels = np.array(all_labels)all_probs = np.array(all_probs)y_bin     = label_binarize(all_labels, classes=list(range(NUM_CLS)))test_acc  = accuracy_score(all_labels, all_preds)macro_auc = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')mcc       = matthews_corrcoef(all_labels, all_preds)bal_acc   = balanced_accuracy_score(all_labels, all_preds)macro_f1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)print(f'ConvNeXt-Tiny v2 (retrained)')print(f'  Test Accuracy : {test_acc*100:.2f}%   (original 71.35%)')print(f'  Macro AUC     : {macro_auc:.4f}       (original 0.9418)')print(f'  MCC           : {mcc:.4f}             (paper claimed 0.6507)')print(f'  Balanced acc  : {bal_acc*100:.2f}%')print(f'  Macro F1      : {macro_f1:.4f}')print('\n' + classification_report(all_labels, all_preds, target_names=CLASSES, digits=4))cm = confusion_matrix(all_labels, all_preds, labels=list(range(NUM_CLS)))# Per-class metrics, same shape NB5/NB6 write, for the paper's Table 3.prec_a, rec_a, f1_a, sup_a = precision_recall_fscore_support(    all_labels, all_preds, labels=list(range(NUM_CLS)), zero_division=0)PER_CLASS = {}for i, c in enumerate(CLASSES):    tp = int(cm[i, i]); fn = int(cm[i].sum() - tp)    fp = int(cm[:, i].sum() - tp); tn = int(cm.sum() - tp - fn - fp)    PER_CLASS[c] = dict(        support=int(sup_a[i]), recall=float(rec_a[i]),        precision=float(prec_a[i]), f1=float(f1_a[i]),        specificity=float(tn / (tn + fp)) if (tn + fp) else 0.0,        auc=float(roc_auc_score((all_labels == i).astype(int),                                all_probs[:, i])))pd.DataFrame(PER_CLASS).T.to_csv(SAVE_DIR/'per_class_test.csv')print('\nper-class:')for c, m in PER_CLASS.items():    print(f"  {c:<24} rec {m['recall']:.4f}  prec {m['precision']:.4f}  "          f"spec {m['specificity']:.4f}  auc {m['auc']:.4f}")pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(SAVE_DIR/'confusion_convnext_v2_counts.csv')pd.DataFrame(cm/cm.sum(1,keepdims=True), index=CLASSES, columns=CLASSES) \  .to_csv(SAVE_DIR/'confusion_convnext_v2_rownorm.csv')save_path = SAVE_DIR/'convnext_tiny_5cls_skin_v2.pth'torch.save({    'model_state'        : model.state_dict(),    'classes'            : CLASSES,    'config'             : CFG,    'best_composite'     : float(best_composite),    'best_val_acc'       : float(best_val_acc),    'best_melanoma_sens' : float(best_mel_sens),    'test_acc'           : float(test_acc),    'macro_auc'          : float(macro_auc),    'mcc'                : float(mcc),    'balanced_accuracy'  : float(bal_acc),    'macro_f1'           : float(macro_f1),    'history'            : history,    'per_class'          : PER_CLASS,    'saved_epoch'        : int(saved_epoch),    'metrics_at_saved_epoch': dict(        val_acc=float(saved_val_acc),        melanoma_sensitivity=float(saved_mel_sens),        bcc_sensitivity=float(saved_bcc_sens)),    'epochs_run'         : epochs_run,    'total_seconds'      : total_seconds,    'gpu'                : GPU_NAME,    'provenance'         : ('Retrain of convnext_tiny_5class.ipynb with '                            'freeze_epochs 8->3, patience 5->10, num_epochs 30->40. '                            'All other settings and seed 42 unchanged.'),}, save_path)print(f'\nsaved {save_path.name}')# ---------------------------------------------------------------------------# Save-then-verify. This is the cell the original checkpoints would have failed:# they stored metrics their own weights could not reproduce (REVISION_NOTES.md# section 0). Reload from disk, rebuild the model, load strict=True, and assert# (1) every tensor is bit-identical to the model just evaluated, (2) re-scoring# reproduces test_acc exactly, (3) the stored value agrees to < 1e-9.print('\n' + '='*72)print('VERIFY: reload the saved checkpoint and re-score it')print('='*72)_ck = torch.load(save_path, map_location='cpu', weights_only=False)assert list(_ck['classes']) == CLASSES, _ck['classes']_m2 = ConvNeXtSkinClassifier(NUM_CLS, _ck['config']['dropout'],                             _ck['config']['drop_path_rate'], pretrained=False)_m2.load_state_dict(_ck['model_state'], strict=True)_m2 = _m2.to(DEVICE, memory_format=torch.channels_last).eval()_sd_mem, _sd_disk = model.state_dict(), _m2.state_dict()assert set(_sd_mem) == set(_sd_disk), 'tensor name sets differ'_n_identical = 0for _k in _sd_mem:    _a, _b = _sd_mem[_k].detach().cpu(), _sd_disk[_k].detach().cpu()    assert _a.shape == _b.shape, (_k, _a.shape, _b.shape)    assert torch.equal(_a, _b), f'tensor {_k} differs between memory and disk'    _n_identical += 1_p2 = []with torch.no_grad():    for imgs, _ in test_loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last,                       non_blocking=True)        _p2.append(torch.softmax(_m2(imgs), dim=1).float().cpu().numpy())_p2 = np.concatenate(_p2)_acc2 = accuracy_score(all_labels, _p2.argmax(1))assert _acc2 == test_acc, f'reloaded acc {_acc2} != in-memory {test_acc}'assert abs(float(_ck['test_acc']) - test_acc) < 1e-9, 'stored test_acc disagrees'VERIFY = dict(passed=True, n_tensors=int(_n_identical),              test_acc_in_memory=float(test_acc),              test_acc_reloaded=float(_acc2),              stored_test_acc=float(_ck['test_acc']))print(f"  tensors compared  : {_n_identical}  (all bit-identical)")print(f"  acc in memory     : {test_acc:.10f}")print(f"  acc reloaded      : {_acc2:.10f}")print(f"  acc stored in ckpt: {float(_ck['test_acc']):.10f}")print(f"  VERIFIED          : {VERIFY['passed']}")del _m2, _sd_mem, _sd_disk, _p2np.savez_compressed(SAVE_DIR/'cache_convnext_tiny_v2_test.npz',                    probs=all_probs, labels=all_labels, classes=np.array(CLASSES))print('saved cache_convnext_tiny_v2_test.npz')

## Re-run the ablation with the retrained ConvNeXtLoads the original EfficientNet and Swin checkpoints, swaps in ConvNeXt v2, andrecomputes the 7-configuration ablation with bootstrap CIs and the pairedensemble-minus-Swin comparison. Writes `results_convnext_v2.json`.The question this answers: **does a fairly trained ConvNeXt change the ablation'sconclusion?**

In [ ]:
# NOTE: these are the RETRAINED (v2) EfficientNet and Swin checkpoints,# from NB5 and NB6. The original *_5cls_skin.pth files do not reproduce# their own stored metrics (REVISION_NOTES.md section 0), so an ablation# built on them would be meaningless. Upload all three v2 checkpoints to# CKPT_ROOT first -- and keep each notebook's output in its own folder,# since the archives share filenames (history.csv, timing.json, ...).CKPT_ROOT = Path(CKPT_ROOT)def _resolve(*cands):    for c in cands:        if Path(c).exists(): return Path(c)    raise FileNotFoundError(cands)class EfficientSkinClassifier(nn.Module):    def __init__(self, num_classes, dropout=0.35, drop_path_rate=0.15):        super().__init__()        self.backbone = timm.create_model('efficientnet_b1', pretrained=False,            num_classes=0, drop_rate=dropout, drop_path_rate=drop_path_rate)        fd = self.backbone.num_features        self.classifier = nn.Sequential(            nn.LayerNorm(fd), nn.Dropout(dropout), nn.Linear(fd, 256),            nn.GELU(), nn.Dropout(dropout/2), nn.Linear(256, num_classes))    def forward(self, x): return self.classifier(self.backbone(x))class SwinSkinClassifier(nn.Module):    def __init__(self, model_name, num_classes, dropout=0.4, drop_path_rate=0.2):        super().__init__()        self.backbone = timm.create_model(model_name, pretrained=False,            num_classes=0, drop_rate=dropout, drop_path_rate=drop_path_rate)        fd = self.backbone.num_features        self.classifier = nn.Sequential(            nn.LayerNorm(fd), nn.Dropout(dropout), nn.Linear(fd, 256),            nn.GELU(), nn.Dropout(dropout/2), nn.Linear(256, num_classes))    def forward(self, x): return self.classifier(self.backbone(x))@torch.no_grad()def infer(m, loader):    out = []    for imgs, _ in loader:        imgs = imgs.to(DEVICE, memory_format=torch.channels_last, non_blocking=True)        out.append(F.softmax(m(imgs), dim=1).float().cpu().numpy())    return np.concatenate(out)ck_eff = torch.load(_resolve(CKPT_ROOT/'efficientnet_b1_5cls_skin_v2.pth',                             CKPT_ROOT/'efficientNet'/'efficientnet_b1_5cls_skin_v2.pth'),                    map_location='cpu', weights_only=False)ck_swin = torch.load(_resolve(CKPT_ROOT/'swin_tiny_5cls_skin_v2.pth',                              CKPT_ROOT/'swin'/'swin_tiny_5cls_skin_v2.pth'),                     map_location='cpu', weights_only=False)assert list(ck_eff['classes']) == CLASSES and list(ck_swin['classes']) == CLASSESm_eff = EfficientSkinClassifier(NUM_CLS, ck_eff['config']['dropout'],                                ck_eff['config']['drop_path_rate'])m_eff.load_state_dict(ck_eff['model_state'], strict=True)m_eff = m_eff.to(DEVICE, memory_format=torch.channels_last).eval()m_swin = SwinSkinClassifier(ck_swin['config']['model_name'], NUM_CLS,                            ck_swin['config']['dropout'], ck_swin['config']['drop_path_rate'])m_swin.load_state_dict(ck_swin['model_state'], strict=True)m_swin = m_swin.to(DEVICE, memory_format=torch.channels_last).eval()print('loaded EfficientNet + Swin (strict=True)')P = {'E': infer(m_eff, test_loader), 'C': all_probs, 'S': infer(m_swin, test_loader)}y = all_labels# sanity: original single-model numbers must still reproduceassert abs(accuracy_score(y, P['E'].argmax(1)) - float(ck_eff['test_acc'])) < 1e-4assert abs(accuracy_score(y, P['S'].argmax(1)) - float(ck_swin['test_acc'])) < 1e-4print('EfficientNet and Swin reproduce their checkpoint accuracies: OK')CONFIGS = ['E','C','S','EC','ES','CS','ECS']def cprobs(cfg): return np.mean([P[k] for k in cfg], axis=0)def mauc(p, yy):    return roc_auc_score(label_binarize(yy, classes=list(range(NUM_CLS))),                         p, multi_class='ovr', average='macro')N_BOOT = 2000rng = np.random.default_rng(SEED)BIDX = rng.integers(0, len(y), size=(N_BOOT, len(y)))res, boot = {}, {}print(f'\n{"config":8s} {"acc":>9} {"macroAUC":>10} {"MCC":>9}   (with ConvNeXt v2)')print('-'*52)for cfg in CONFIGS:    p = cprobs(cfg); pr = p.argmax(1)    res[cfg] = dict(accuracy=float(accuracy_score(y, pr)), macro_auc=float(mauc(p, y)),                    mcc=float(matthews_corrcoef(y, pr)),                    balanced_accuracy=float(balanced_accuracy_score(y, pr)),                    macro_f1=float(f1_score(y, pr, average='macro', zero_division=0)),                    confusion_matrix=confusion_matrix(y, pr, labels=list(range(NUM_CLS))).tolist())    accs, mccs, aucs = [], [], []    for idx in BIDX:        yb = y[idx]        accs.append((yb == pr[idx]).mean()); mccs.append(matthews_corrcoef(yb, pr[idx]))        if len(np.unique(yb)) == NUM_CLS: aucs.append(mauc(p[idx], yb))    q = lambda a: dict(lo=float(np.percentile(a,2.5)), hi=float(np.percentile(a,97.5)),                       mean=float(np.mean(a)), n=len(a))    boot[cfg] = dict(accuracy=q(accs), mcc=q(mccs), macro_auc=q(aucs))    print(f'{cfg:8s} {res[cfg]["accuracy"]*100:>8.2f}% {res[cfg]["macro_auc"]:>10.4f} '          f'{res[cfg]["mcc"]:>9.4f}')pa, pb = cprobs('ECS').argmax(1), cprobs('S').argmax(1)d_acc = np.array([ (y[i]==pa[i]).mean() - (y[i]==pb[i]).mean() for i in BIDX ])paired = dict(accuracy_pp=dict(    observed=float((res['ECS']['accuracy']-res['S']['accuracy'])*100),    ci_lo=float(np.percentile(d_acc,2.5)*100), ci_hi=float(np.percentile(d_acc,97.5)*100),    ci_includes_zero=bool(np.percentile(d_acc,2.5) <= 0 <= np.percentile(d_acc,97.5))))print(f'\npaired {{E,C,S}} - Swin accuracy: {paired["accuracy_pp"]["observed"]:+.3f} pp '      f'CI [{paired["accuracy_pp"]["ci_lo"]:+.3f}, {paired["accuracy_pp"]["ci_hi"]:+.3f}]'      f'{"  (includes 0)" if paired["accuracy_pp"]["ci_includes_zero"] else ""}')with open(SAVE_DIR/'results_convnext_v2.json','w',encoding='utf-8') as f:    json.dump(dict(        meta=dict(notebook='04_convnext_retrain.ipynb', seed=SEED, gpu=GPU_NAME,                  n_test=int(len(y)), n_boot=N_BOOT,                  convnext_changes=dict(freeze_epochs=[8,3], patience=[5,10], num_epochs=[30,40]),                  note='ConvNeXt v2 replaces the original under-trained ConvNeXt. '                       'EfficientNet and Swin checkpoints are unchanged.'),        convnext_v2=dict(epochs_run=epochs_run, test_acc=float(test_acc),                         macro_auc=float(macro_auc), mcc=float(mcc),                         best_val_acc=float(best_val_acc),                         total_hours=total_seconds/3600),        convnext_original=dict(epochs_run=9, test_acc=0.7134831460674157,                               macro_auc=0.9417740627766318, best_val_acc=0.7556818181818182),        loss_weights=LOSS_WEIGHTS,        configs={c: dict(metrics=res[c], bootstrap_ci=boot[c]) for c in CONFIGS},        paired_ensemble_minus_swin=paired,    ), f, indent=2)print('\nwrote results_convnext_v2.json')print('\nCompare against results.json from Notebook 1 to see whether a fairly')print('trained ConvNeXt changes the ablation conclusion.')